[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/clase/S4_svm_comparativa.ipynb) · [Ver en el sitio](https://hesusg.github.io/dammg-supervisado/notebooks/clase/S4_svm_comparativa.html) · [Menú de notebooks](https://hesusg.github.io/dammg-supervisado/notebooks/)

# Sesión 4 · SVM y la comparativa final, en chiquito
**DAMMG · Módulo VI · sáb 7 nov** · 4 mini-labs de unos 15–20 min cada uno · **no se entrega ni se califica**

| # | Mini-lab | Idea clave |
|---|---|---|
| 6 | La calle más ancha | Entre todas las rectas que separan, la SVM elige la que deja más espacio |
| 7 | C: cuánto error tolero | C chico: calle ancha con errores. C grande: calle angosta, sin perdonar |
| 8 | Kernel y γ | Si una recta no alcanza, agrega una dimensión donde sí alcance |
| 9 | Todos contra todos | Cada modelo hace una apuesta distinta sobre los datos |

> **Cómo usar este notebook**
> 1. *Archivo → Guardar una copia en Drive* y trabaja en tu copia.
> 2. Ejecuta las celdas **en orden** con `Shift + Enter`.
> 3. Solo cambias código en las celdas marcadas **✋ Tu turno**.
> 4. Las preguntas **💬 Para el chat** se responden en el chat de Zoom, con una frase basta.
>
> Si algo falla: *Entorno de ejecución → Reiniciar y ejecutar todo*.

In [ ]:
# Preparación: ejecuta esta celda una vez. No necesitas leerla.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

sns.set_theme(style="whitegrid", context="notebook")
AZUL, ROJO = "#4C72B0", "#C44E52"


def dibuja_frontera(modelo, X, y, ax=None, titulo="", nombres=("clase 0", "clase 1")):
    """Pinta de color la zona que el modelo asigna a cada clase y encima los puntos."""
    ax = ax or plt.gca()
    c0, c1 = X.columns[:2]
    xx, yy = np.meshgrid(
        np.linspace(X[c0].min() - 0.5, X[c0].max() + 0.5, 250),
        np.linspace(X[c1].min() - 0.5, X[c1].max() + 0.5, 250),
    )
    malla = pd.DataFrame({c0: xx.ravel(), c1: yy.ravel()})
    zona = modelo.predict(malla).reshape(xx.shape)
    ax.contourf(xx, yy, zona, levels=[-0.5, 0.5, 1.5], cmap=ListedColormap([AZUL, ROJO]), alpha=0.2)
    etiquetas = pd.Series(np.asarray(y)).map({0: nombres[0], 1: nombres[1]})
    sns.scatterplot(x=X[c0].values, y=X[c1].values, hue=etiquetas.values, hue_order=list(nombres),
                    palette={nombres[0]: AZUL, nombres[1]: ROJO}, s=60, edgecolor="k", ax=ax)
    ax.set(xlabel=c0, ylabel=c1, title=titulo)
    return ax


def dibuja_calle(svm, X, y, ax=None, titulo=""):
    """Dibuja la frontera (línea continua), los bordes de la calle (punteadas)
    y encierra en un círculo los vectores de soporte."""
    ax = dibuja_frontera(svm, X, y, ax=ax, titulo=titulo)
    c0, c1 = X.columns[:2]
    xx, yy = np.meshgrid(
        np.linspace(X[c0].min() - 0.5, X[c0].max() + 0.5, 250),
        np.linspace(X[c1].min() - 0.5, X[c1].max() + 0.5, 250),
    )
    malla = pd.DataFrame({c0: xx.ravel(), c1: yy.ravel()})
    z = svm.decision_function(malla).reshape(xx.shape)
    ax.contour(xx, yy, z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=220, facecolors="none", edgecolors="k", linewidths=1.5)
    return ax

print("✅ entorno listo")

---
## 6 · La calle más ancha

🎯 **Idea clave.** Muchas rectas separan dos grupos. La SVM elige la que deja la **calle más ancha** entre ellos. Los puntos que tocan el borde de la calle se llaman **vectores de soporte**: son los únicos que importan. Los demás podrían no estar y la calle sería la misma.

30 puntos en dos grupos bien separados:

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

Xb, yb = make_blobs(n_samples=30, centers=2, cluster_std=0.9, random_state=6)
Xb = pd.DataFrame(Xb, columns=["x1", "x2"])

svm = SVC(kernel="linear", C=1000).fit(Xb, yb)
dibuja_calle(svm, Xb, yb, titulo="La calle: frontera (continua) y bordes (punteadas)");
print("Vectores de soporte (encerrados en círculo):", len(svm.support_))

### 6a · ¿Qué tan ancha es la calle?
La SVM aprende un vector de pesos $w$. El ancho de la calle es $\frac{2}{\lVert w \rVert}$: entre más chicos los pesos, más ancha la calle.

In [ ]:
w = svm.coef_[0]
print("w =", w.round(3))
print("ancho de la calle =", round(2 / np.linalg.norm(w), 3))

### 6b · Comprueba que solo importan los vectores de soporte
Quitamos un punto **lejano** (que no es vector de soporte) y volvemos a entrenar. Después quitamos un **vector de soporte**.

✋ **Tu turno.** Ejecuta y compara los dos anchos. Luego cambia `cual` de 0 a 1 para quitar el otro vector de soporte.

In [ ]:
soporte = svm.support_
lejano = [i for i in Xb.index if i not in soporte][0]
cual = 0                                       # ✋ CAMBIA: 0 o 1

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for ax, quitar, texto in zip(ejes, [lejano, soporte[cual]], ["sin un punto lejano", "sin un vector de soporte"]):
    Xq, yq = Xb.drop(index=quitar), np.delete(yb, quitar)
    s = SVC(kernel="linear", C=1000).fit(Xq, yq)
    dibuja_calle(s, Xq, yq, ax=ax, titulo=f"{texto}\nancho = {2 / np.linalg.norm(s.coef_[0]):.3f}")
plt.tight_layout()

💬 **Para el chat.** ¿Cambió el ancho al quitar el punto lejano? ¿Y al quitar el vector de soporte? ¿Qué te dice eso de los puntos lejanos?

---
## 7 · C: cuánto error tolero

🎯 **Idea clave.** En datos reales las clases se enciman y ninguna calle queda limpia. El parámetro **C** es el precio de cada punto que cae dentro de la calle o del lado equivocado.
- **C chico:** el error sale barato → calle **ancha**, tolera errores, muchos vectores de soporte.
- **C grande:** el error sale caro → calle **angosta**, intenta acertar a todos, puede sobreajustar.

In [ ]:
from sklearn.model_selection import train_test_split

Xc, yc = make_blobs(n_samples=120, centers=[[0, 0], [3, 3]], cluster_std=1.6, random_state=2)
Xc = pd.DataFrame(Xc, columns=["x1", "x2"])
Xc_ent, Xc_prueba, yc_ent, yc_prueba = train_test_split(Xc, yc, test_size=0.3, random_state=0)

✋ **Tu turno.** Cambia los tres valores de `C` (prueba 0.001, 0.1, 10, 1000).

In [ ]:
valores_C = [0.01, 1, 100]                 # ✋ CAMBIA ESTOS NÚMEROS

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, C in zip(ejes, valores_C):
    s = SVC(kernel="linear", C=C).fit(Xc_ent, yc_ent)
    dibuja_calle(s, Xc_ent, yc_ent, ax=ax,
                 titulo=f"C = {C} · soporte: {len(s.support_)} · prueba: {s.score(Xc_prueba, yc_prueba):.0%}")
plt.tight_layout()

💬 **Para el chat.** Al subir C, ¿qué le pasa al ancho de la calle y al número de vectores de soporte? ¿Cambió mucho la exactitud en prueba?

---
## 8 · Kernel y γ: cuando una recta no alcanza

🎯 **Idea clave.** Hay datos que ninguna recta separa, como un círculo dentro de otro. El truco: **agregar una columna nueva** donde sí se puedan separar. Un **kernel** hace eso por ti sin calcular la columna.

### 8a · Una recta fracasa

In [ ]:
from sklearn.datasets import make_circles

Xa, ya = make_circles(n_samples=200, factor=0.4, noise=0.2, random_state=0)
Xa = pd.DataFrame(Xa, columns=["x1", "x2"])
Xa_ent, Xa_prueba, ya_ent, ya_prueba = train_test_split(Xa, ya, test_size=0.3, random_state=0)

lineal = SVC(kernel="linear").fit(Xa_ent, ya_ent)
dibuja_frontera(lineal, Xa_ent, ya_ent, titulo=f"SVM lineal · prueba: {lineal.score(Xa_prueba, ya_prueba):.0%}");

### 8b · Agrega una columna a mano
Calculamos la distancia al centro al cuadrado, $x_1^2 + x_2^2$. Los de adentro dan números chicos y los de afuera, grandes. Mira lo fácil que se separan ahora:

In [ ]:
Xa_ent = Xa_ent.assign(r2=Xa_ent["x1"]**2 + Xa_ent["x2"]**2)
Xa_prueba = Xa_prueba.assign(r2=Xa_prueba["x1"]**2 + Xa_prueba["x2"]**2)

sns.stripplot(x=ya_ent, y=Xa_ent["r2"], hue=ya_ent, palette={0: AZUL, 1: ROJO}, legend=False, size=6)
plt.xlabel("clase"); plt.ylabel("x1² + x2²")
plt.title("En la columna nueva, un solo corte separa todo");

lineal3 = SVC(kernel="linear").fit(Xa_ent, ya_ent)
print(f"SVM lineal con la columna nueva · prueba: {lineal3.score(Xa_prueba, ya_prueba):.0%}")

### 8c · El kernel RBF lo hace solo, y γ controla qué tan "curva" es
Con `kernel="rbf"` no agregas columnas: la SVM mide parecido entre puntos y dobla la frontera sola. El parámetro **γ** (gamma) decide qué tan lejos llega la influencia de cada punto.
- **γ chico:** cada punto influye lejos → frontera **suave**.
- **γ grande:** cada punto solo influye a su alrededor → frontera de **islitas**, memoriza.

✋ **Tu turno.** Cambia los valores de `gamma` (prueba 0.01, 0.5, 5, 100).

In [ ]:
Xa_ent, Xa_prueba = Xa_ent[["x1", "x2"]], Xa_prueba[["x1", "x2"]]   # volvemos a 2 columnas
valores_gamma = [0.1, 1, 500]          # ✋ CAMBIA ESTOS NÚMEROS

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, g in zip(ejes, valores_gamma):
    s = SVC(kernel="rbf", gamma=g).fit(Xa_ent, ya_ent)
    dibuja_frontera(s, Xa_ent, ya_ent, ax=ax,
                    titulo=f"γ = {g} · entrenamiento: {s.score(Xa_ent, ya_ent):.0%} · prueba: {s.score(Xa_prueba, ya_prueba):.0%}")
plt.tight_layout()

> **Ojo con las escalas.** Igual que KNN, la SVM mide distancias. Con datos reales, escala antes (`StandardScaler`).

💬 **Para el chat.** Con γ = 500, ¿por qué baja la exactitud en prueba aunque en entrenamiento sea casi perfecta?

---
## 9 · Todos contra todos

🎯 **Idea clave.** Ningún modelo gana siempre. Cada uno hace una **apuesta** sobre cómo son los datos (su *sesgo inductivo*). Gana el que apuesta mejor para *este* problema, y eso solo lo dices con datos de **prueba**.

| Modelo | Su apuesta |
|---|---|
| Logística | La frontera es una recta |
| KNN | Lo cercano se parece |
| Árbol | Se puede decidir con preguntas de sí o no, en escalones |
| Bosque | Muchos árboles votando se equivocan menos que uno |
| SVM (RBF) | La mejor frontera deja el mayor margen, aunque sea curva |

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons

ruido = 0.3                     # ✋ al final cambiarás este número

Xm, ym = make_moons(n_samples=300, noise=ruido, random_state=0)
Xm = pd.DataFrame(Xm, columns=["x1", "x2"])
Xm_ent, Xm_prueba, ym_ent, ym_prueba = train_test_split(Xm, ym, test_size=0.3, random_state=0)

modelos = {
    "Logística": LogisticRegression(),
    "KNN (k=15)": KNeighborsClassifier(n_neighbors=15),
    "Árbol (prof. 4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "Bosque": RandomForestClassifier(n_estimators=200, random_state=0),
    "SVM (RBF)": SVC(kernel="rbf", C=1, gamma=1),
}

### 9a · Las fronteras de todos

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(16, 9))
resultados = []
for ax, (nombre, m) in zip(ejes.ravel(), modelos.items()):
    m.fit(Xm_ent, ym_ent)
    resultados.append({"modelo": nombre,
                       "entrenamiento": m.score(Xm_ent, ym_ent),
                       "prueba": m.score(Xm_prueba, ym_prueba)})
    dibuja_frontera(m, Xm_ent, ym_ent, ax=ax, titulo=f"{nombre} · prueba: {resultados[-1]['prueba']:.0%}")
ejes.ravel()[-1].axis("off")
plt.tight_layout()

### 9b · La tabla y la gráfica

In [ ]:
res = pd.DataFrame(resultados).sort_values("prueba", ascending=False)
largo = res.melt(id_vars="modelo", var_name="conjunto", value_name="exactitud")

sns.barplot(data=largo, x="exactitud", y="modelo", hue="conjunto")
plt.xlim(0.5, 1.0); plt.title("¿Quién gana en prueba?")
res.round(2)

✋ **Tu turno.** Sube a la celda de arriba (9) y cambia `ruido` a **0.1** y luego a **0.5**. Ejecuta 9, 9a y 9b otra vez. ¿Cambia el ganador? ¿Cuál sufre más con el ruido?

💬 **Para el chat.** Si dos modelos difieren por 1 o 2 puntos de exactitud en 90 datos de prueba, ¿escogerías el "ganador" o el más sencillo? ¿Por qué?

---
### Para llevarte
1. La **SVM** busca la calle más ancha; solo importan los **vectores de soporte**.
2. **C** y **γ** controlan cuánto se ajusta: valores grandes memorizan.
3. Un **kernel** separa con una curva porque, por dentro, agrega dimensiones.
4. No hay modelo ganador universal: **compara en prueba** y explica la apuesta de cada uno.

Lo que sigue: **Lab 4** (se entrega el dom 15 nov) y el **proyecto final** (dom 22 nov).